In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (

    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import LSTM
from tensorflow.keras.optimizers import (
    Adam,
    RMSprop
)

In [4]:
data=pd.read_csv('/kaggle/input/datasets/manishavelamani/dataset/PJME_preprocessd.csv',parse_dates=['Datetime'],index_col='Datetime')

In [5]:
data.columns

Index(['PJME_MW', 'PJME_MW_Scaled', 'Hour', 'Day', 'Week', 'Month',
       'DayOfWeek', 'Weekend', 'Lag_1', 'Lag_24', 'Lag_48', 'Lag_168',
       'RollingMean_24', 'RollingStd_24', 'RollingMean_168'],
      dtype='object')

In [6]:
from sklearn.preprocessing import MinMaxScaler

features = [
    'PJME_MW_Scaled',
    'Hour',
    'Day',
    'Week',
    'Month',
    'DayOfWeek',
    'Weekend',
    'Lag_1',
    'Lag_24',
    'Lag_48',
    'Lag_168',
    'RollingMean_24',
    'RollingStd_24',
    'RollingMean_168'
]

target = 'PJME_MW_Scaled'

# Select input features
multivariate_data = data[features].copy()

# Scale ALL input features
feature_scaler = MinMaxScaler()

multivariate_data_scaled = pd.DataFrame(
    feature_scaler.fit_transform(multivariate_data),
    columns=features,
    index=multivariate_data.index
)

multivariate_data_scaled.head()

,PJME_MW_Scaled,Hour,Day,Week,Month,DayOfWeek,Weekend,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingStd_24,RollingMean_168
Datetime,,,,,,,,,,,,,,
2002-01-08 01:00:00,0.433011,0.043478,0.233333,0.019231,0.0,0.166667,0.0,0.486937,0.353052,0.360420,0.462358,0.535733,0.407201,0.421362
2002-01-08 02:00:00,0.409021,0.086957,0.233333,0.019231,0.0,0.166667,0.0,0.433011,0.325625,0.329371,0.427439,0.539989,0.386780,0.421179
2002-01-08 03:00:00,0.399889,0.130435,0.233333,0.019231,0.0,0.166667,0.0,0.409021,0.315255,0.319960,0.399331,0.544309,0.363683,0.421185
2002-01-08 04:00:00,0.405058,0.173913,0.233333,0.019231,0.0,0.166667,0.0,0.399889,0.316029,0.315750,0.385154,0.548853,0.338064,0.421382
2002-01-08 05:00:00,0.427316,0.217391,0.233333,0.019231,0.0,0.166667,0.0,0.405058,0.336522,0.319496,0.390045,0.553487,0.312793,0.421752


In [7]:
train_size = int(len(multivariate_data_scaled) * 0.70)
val_size = int(len(multivariate_data_scaled) * 0.10)

train = multivariate_data_scaled.iloc[:train_size]
validation = multivariate_data_scaled.iloc[train_size:train_size + val_size]
test = multivariate_data_scaled.iloc[train_size + val_size:]

In [8]:
def create_multivariate_sequences(df, sequence_length, forecast_horizon=24, target_col='PJME_MW_Scaled'):
    X = []
    y = []

    values = df.values
    target_index = df.columns.get_loc(target_col)

    for i in range(len(df) - sequence_length - forecast_horizon + 1):
        X.append(values[i:i + sequence_length])

        y.append(
            values[
                i + sequence_length:
                i + sequence_length + forecast_horizon,
                target_index
            ]
        )

    return np.array(X), np.array(y)

In [9]:
sequence_length = 168

X_train, y_train = create_multivariate_sequences(train, sequence_length)
X_val, y_val = create_multivariate_sequences(validation, sequence_length)
X_test, y_test = create_multivariate_sequences(test, sequence_length)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_val:', X_val.shape)
print('y_val:', y_val.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (101465, 168, 14)
y_train: (101465, 24)
X_val: (14331, 168, 14)
y_val: (14331, 24)
X_test: (28855, 168, 14)
y_test: (28855, 24)


In [10]:
from sklearn.preprocessing import MinMaxScaler
# Create scaler using the original MW values
scaler = MinMaxScaler()
scaler.fit(data[['PJME_MW']])

MinMaxScaler()

In [11]:
def evaluate_model(model, X_test, y_test, scaler, sequence_length, model_name):
    predictions = model.predict(X_test, verbose=0)
    # Convert scaled values back to original MW values
    pred_original = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    y_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    mae = mean_absolute_error(y_original.flatten(),pred_original.flatten())
    mse = mean_squared_error(y_original.flatten(),pred_original.flatten())
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_original.flatten(),pred_original.flatten()) * 100
    r2 = r2_score(y_original.flatten(),pred_original.flatten())
    bias = np.mean(pred_original.flatten() - y_original.flatten())
    return {
        'Sequence Length': sequence_length,
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Bias': bias
    }

In [12]:
lstm_phase6 = Sequential([
    LSTM(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_phase6.compile(
    optimizer='adam',
    loss='mse'
)

lstm_phase6.summary()

lstm_baseline_history = lstm_phase6.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

baseline_lstm_result = evaluate_model(
    lstm_phase6,
    X_test,
    y_test,
    scaler,
    168,
    'Baseline LSTM (Multivariate)'
)

I0000 00:00:1786460609.114385      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         6,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,808 (26.59 KB)

 Trainable params: 6,808 (26.59 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 33s 10ms/step - loss: 0.0080 - val_loss: 0.0040
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0032 - val_loss: 0.0038
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0028 - val_loss: 0.0032
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0027 - val_loss: 0.0032
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0026 - val_loss: 0.0034
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0025 - val_loss: 0.0032


In [14]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

lstm_early = Sequential([
    LSTM(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_early.compile(
    optimizer='adam',
    loss='mse'
)

lstm_early_history = lstm_early.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

lstm_early_result = evaluate_model(
    lstm_early,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + EarlyStopping'
)

Epoch 1/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 9ms/step - loss: 0.0088 - val_loss: 0.0043
Epoch 2/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0033 - val_loss: 0.0038
Epoch 3/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 4/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0029 - val_loss: 0.0036
Epoch 5/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 6/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0027 - val_loss: 0.0034
Epoch 7/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0027 - val_loss: 0.0032
Epoch 8/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0026 - val_loss: 0.0033
Epoch 9/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 10/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0025 - val_loss: 0.0033
Epoch 11/20
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 12/20
3171/31

In [15]:
lstm_dropout = Sequential([
    LSTM(32, input_shape=(sequence_length, X_train.shape[2])),
    Dropout(0.2),
    Dense(24)
])

lstm_dropout.compile(
    optimizer='adam',
    loss='mse'
)

lstm_dropout_history = lstm_dropout.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

lstm_dropout_result = evaluate_model(
    lstm_dropout,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + Dropout'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 9ms/step - loss: 0.0133 - val_loss: 0.0043
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0047 - val_loss: 0.0038
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0043 - val_loss: 0.0037
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0042 - val_loss: 0.0036
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0040 - val_loss: 0.0035
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0040 - val_loss: 0.0034
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0039 - val_loss: 0.0033
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0039 - val_loss: 0.0032
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0038 - val_loss: 0.0034
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0038 - val_loss: 0.0032


In [16]:
lstm_more_units = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_more_units.compile(
    optimizer='adam',
    loss='mse'
)

lstm_more_units_history = lstm_more_units.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
   
)

lstm_more_units_result = evaluate_model(
    lstm_more_units,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + More Units'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0068 - val_loss: 0.0039
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0032 - val_loss: 0.0036
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0028 - val_loss: 0.0032
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0027 - val_loss: 0.0035
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0024 - val_loss: 0.0032
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0023 - val_loss: 0.0030
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0022 - val_loss: 0.0030


In [17]:
lstm_batchnorm = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    BatchNormalization(),
    Dense(24)
])

lstm_batchnorm.compile(
    optimizer='adam',
    loss='mse'
)

lstm_batchnorm_history = lstm_batchnorm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

lstm_batchnorm_result = evaluate_model(
    lstm_batchnorm,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + BatchNormalization'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - loss: 0.0173 - val_loss: 0.0068
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0053 - val_loss: 0.0081
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0048 - val_loss: 0.0061
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0045 - val_loss: 0.0055
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0043 - val_loss: 0.0043
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0042 - val_loss: 0.0048
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0041 - val_loss: 0.0042
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0040 - val_loss: 0.0037
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0040 - val_loss: 0.0036
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0039 - val_loss: 0.0035


In [18]:
lstm_rmsprop = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='mse'
)

lstm_rmsprop_history = lstm_rmsprop.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

lstm_rmsprop_result = evaluate_model(
    lstm_rmsprop,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + RMSprop'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0064 - val_loss: 0.0040
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0034 - val_loss: 0.0038
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0031 - val_loss: 0.0039
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0030 - val_loss: 0.0038
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0029 - val_loss: 0.0037
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0026 - val_loss: 0.0033
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0026 - val_loss: 0.0032


In [19]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

lstm_lr = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_lr.compile(
    optimizer='adam',
    loss='mse'
)

lstm_lr_history = lstm_lr.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop, lr_scheduler]
)

lstm_lr_result = evaluate_model(
    lstm_lr,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + LR Scheduler'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0060 - val_loss: 0.0041 - learning_rate: 0.0010
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0031 - val_loss: 0.0035 - learning_rate: 0.0010
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0029 - val_loss: 0.0035 - learning_rate: 0.0010


In [22]:
lstm_bs16 = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_bs16.compile(
    optimizer='adam',
    loss='mse'
)

lstm_bs16_history = lstm_bs16.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16
)

lstm_bs16_result = evaluate_model(
    lstm_bs16,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + Batch Size 16'
)

Epoch 1/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0050 - val_loss: 0.0036
Epoch 2/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0030 - val_loss: 0.0036
Epoch 3/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 4/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 54s 9ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 5/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0025 - val_loss: 0.0030
Epoch 6/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 54s 9ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 7/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0023 - val_loss: 0.0029
Epoch 8/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 52s 8ms/step - loss: 0.0022 - val_loss: 0.0031
Epoch 9/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 52s 8ms/step - loss: 0.0021 - val_loss: 0.0029
Epoch 10/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 52s 8ms/step - loss: 0.0021 - val_loss: 0.0030


In [23]:
lstm_bs64 = Sequential([
    LSTM(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

lstm_bs64.compile(
    optimizer='adam',
    loss='mse'
)

lstm_bs64_history = lstm_bs64.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

lstm_bs64_result = evaluate_model(
    lstm_bs64,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + Batch Size 64'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0083 - val_loss: 0.0047
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0034 - val_loss: 0.0040
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 9/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 10/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0024 - val_loss: 0.0033


In [24]:
lstm_two_layers = Sequential([
    LSTM(64, return_sequences=True, input_shape=(sequence_length, X_train.shape[2])),
    LSTM(32),
    Dense(24)
])

lstm_two_layers.compile(
    optimizer='adam',
    loss='mse'
)

lstm_two_layers.summary()

lstm_two_layers_history = lstm_two_layers.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

lstm_two_layers_result = evaluate_model(
    lstm_two_layers,
    X_test,
    y_test,
    scaler,
    168,
    'LSTM + Two Layers'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_12 (LSTM)                  │ (None, 168, 64)        │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,432 (130.59 KB)

 Trainable params: 33,432 (130.59 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.0072 - val_loss: 0.0042
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0027 - val_loss: 0.0034
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0025 - val_loss: 0.0032
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0024 - val_loss: 0.0033
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0023 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0022 - val_loss: 0.0031
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0021 - val_loss: 0.0031
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0020 - val_loss: 0.0031
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0019 - val_loss: 0.0032


In [25]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
 
# ---------------------------------------------------
# Build LSTM model for hyperparameter tuning
# ---------------------------------------------------
def build_lstm(hp):
 
    model = Sequential()
 
    model.add(
        LSTM(
            units=hp.Choice(
                'lstm_units',
                values=[32, 64, 128]
            ),
            input_shape=(sequence_length, X_train.shape[2])
        )
    )
 
    model.add(
        Dropout(
            hp.Choice(
                'dropout_rate',
                values=[0.0, 0.2, 0.3]
            )
        )
    )
 
    model.add(
        Dense(
            units=hp.Choice(
                'dense_units',
                values=[32, 64, 128]
            ),
            activation='relu'
        )
    )
 
    model.add(Dense(24))
 
    learning_rate = hp.Choice(
        'learning_rate',
        values=[0.0001, 0.0005, 0.001]
    )
 
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss='mse',
        metrics=['mae']
    )
 
    return model
 
 
# ---------------------------------------------------
# Hyperparameter search
# ---------------------------------------------------
tuner = kt.RandomSearch(
    build_lstm,
    objective='val_loss',
    max_trials=3,
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='lstm_168_to_24'
)
 
tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
# ---------------------------------------------------
# Get best hyperparameters
# ---------------------------------------------------
best_hp = tuner.get_best_hyperparameters(1)[0]
 
print('Best LSTM Units:', best_hp.get('lstm_units'))
print('Best Dropout:', best_hp.get('dropout_rate'))
print('Best Dense Units:', best_hp.get('dense_units'))
print('Best Learning Rate:', best_hp.get('learning_rate'))
 
 
# ---------------------------------------------------
# Rebuild and retrain the best model
# ---------------------------------------------------
final_lstm = build_lstm(best_hp)
 
final_lstm_history = final_lstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
 
 
# ---------------------------------------------------
# Evaluate the retrained final model
# ---------------------------------------------------
final_lstm_result = evaluate_model(
    final_lstm,
    X_test,
    y_test,
    scaler,
    168,
    'Final Tuned LSTM'
)
 
pd.DataFrame([final_lstm_result])
 
 
# ---------------------------------------------------
# Save the final tuned model
# ---------------------------------------------------
final_lstm.save('/kaggle/working/lstm168_phase6_final_tuned.keras')



Trial 3 Complete [00h 01m 53s]
val_loss: 0.003497006604447961

Best val_loss So Far: 0.003497006604447961
Total elapsed time: 00h 05m 13s
Best LSTM Units: 128
Best Dropout: 0.3
Best Dense Units: 64
Best Learning Rate: 0.001
Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - loss: 0.0077 - mae: 0.0627 - val_loss: 0.0042 - val_mae: 0.0484
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0038 - mae: 0.0460 - val_loss: 0.0040 - val_mae: 0.0462
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 36s 11ms/step - loss: 0.0034 - mae: 0.0435 - val_loss: 0.0035 - val_mae: 0.0425


In [26]:
comparison_lstm = pd.DataFrame([
    baseline_lstm_result,
    lstm_early_result,
    lstm_dropout_result,
    lstm_more_units_result,
    lstm_batchnorm_result,
    lstm_rmsprop_result,
    lstm_lr_result,
    lstm_bs16_result,
    lstm_bs64_result,
    lstm_two_layers_result,
    final_lstm_result
])

comparison_lstm = comparison_lstm.sort_values('RMSE').reset_index(drop=True)
comparison_lstm

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,LSTM + EarlyStopping,1296.983477,3.311891e+06,1819.860119,4.114061,0.917045,-9.866167
1,168,LSTM + Dropout,1353.550113,3.545038e+06,1882.827218,4.271911,0.911205,-128.659747
2,168,LSTM + Batch Size 16,1334.469027,3.559146e+06,1886.569817,4.261476,0.910851,306.619994
3,168,LSTM + More Units,1354.712617,3.584876e+06,1893.376768,4.347742,0.910207,358.222134
4,168,Baseline LSTM (Multivariate),1361.539889,3.620224e+06,1902.688728,4.383505,0.909322,409.397589
5,168,LSTM + RMSprop,1357.516686,3.625902e+06,1904.180051,4.322344,0.909179,10.355606
6,168,LSTM + Batch Size 64,1393.221496,3.716355e+06,1927.784871,4.466757,0.906914,316.501319
7,168,LSTM + BatchNormalization,1388.175084,3.767326e+06,1940.960163,4.439051,0.905637,247.997014
8,168,LSTM + Two Layers,1436.194281,4.108680e+06,2026.987976,4.605868,0.897087,447.195338
9,168,LSTM + LR Scheduler,1490.343173,4.264251e+06,2065.006401,4.703238,0.893190,-237.847082


In [27]:
comparison_lstm.to_csv('/kaggle/working/Comparisonlstm_csv')
print("saved")

saved
